# SkillCompass: End-to-End Modular Machine Learning Pipeline
## Systematic Submodule Documentation & Execution Notebook

**Author:** Senior Data Science Engineer  
**Architecture:** Modular Python Packages (`src/preprocessing`, `src/feature_engineering`, `src/salary_model`, `src/skill_analyzer`, `src/company_match`)  
**Environment:** Python 3.12+ / 3.14  

---

### Overview
This master notebook integrates, documents, and executes each modular submodule under dedicated headings. Each section describes the submodule's design rationale, architectural boundaries, data transformations, and explicit verification steps.

## System Architecture & Modular Submodule Blueprint

`
EduPredict_AI / SkillCompass Architecture:
├── data/
│   ├── raw/                           # Raw input records (read-only)
│   ├── processed/                     # Cleaned dataset & audit trail logs
│   └── external/                      # Reference dictionaries & skill mappings
├── models/                            # Serialized production bundles (.joblib)
├── reports/                           # Performance figures, confusion matrices, audit logs
├── src/                               # Modular application source code
│   ├── preprocessing/                 # Ingestion, validation, cleaning, and imputation
│   ├── features/                      # Custom transformers & scikit-learn preprocessors
│   ├── models/                        # Cross-validation, hyperparameter tuning & bundling
│   ├── evaluation/                    # Classification/regression metrics & explainability
│   └── service/                       # REST API & Dashboard business adapters
├── api/                               # FastAPI application endpoints
├── dashboard/                         # Streamlit interactive portal
├── notebooks/                         # Diagnostic and exploratory Jupyter notebooks
└── tests/                             # Automated unit, integration, & regression tests
`

---

### Detailed Module & Submodule Specifications

#### Module 1: Ingestion, Validation & Preprocessing (src/preprocessing/)
* **ingestion.py**:
  * *Operations*: Ingests raw CSV partitions, handles character encoding, and checks schema presence.
  * *Input/Output*: Ingests data/raw/*.csv $\rightarrow$ returns merged raw pd.DataFrame.
* **cleaner.py**:
  * *Operations*: Removes exact duplicates, canonicalizes mixed-case strings, and normalizes categorical aliases.
  * *Input/Output*: Raw pd.DataFrame $\rightarrow$ cleaned pd.DataFrame and audit log list.
* **alidator.py**:
  * *Operations*: Validates academic/experience ranges ($[0, 100]\%$, $[0, 30]$ years, non-negative scores) and flags anomalies.
  * *Input/Output*: Cleaned pd.DataFrame $\rightarrow$ validated pd.DataFrame with anomaly codes.
* **imputer.py**:
  * *Operations*: Imputes missing values with stratified medians/modes and exports cleaned_data.csv and cleaning_log.md.
  * *Input/Output*: Validated pd.DataFrame $\rightarrow$ data/processed/cleaned_data.csv.

#### Module 2: Feature Engineering & Preprocessing Pipelines (src/features/)
* **cademic_transformers.py / eature_engineering.py**:
  * *Operations*: Computes score trajectories and experience polynomial terms (, x^2, hi - lo$).
  * *Input/Output*: Raw columns $\rightarrow$ engineered numerical feature matrix.
* **ehavioral_encoder.py**:
  * *Operations*: Multi-hot encodes delimited activities and tool tags on training data.
  * *Input/Output*: Delimited text Series $\rightarrow$ multi-hot indicator matrix.
* **pipeline_builder.py**:
  * *Operations*: Combines OneHotEncoder, RobustScaler, and custom feature transformers into a single scikit-learn ColumnTransformer.
  * *Input/Output*: Training partition $\rightarrow$ fitted ColumnTransformer.

#### Module 3: Model Architecture, Training & Selection (src/models/)
* **candidate_pool.py**:
  * *Operations*: Instantiates regression/classification algorithms (Ridge, Decision Trees, Random Forest, Gradient Boosting, XGBoost).
  * *Input/Output*: Hyperparameters $\rightarrow$ dictionary of candidate estimators.
* **	rainer.py / salary_model.py**:
  * *Operations*: Executes 5-fold cross-validation on log-transformed targets, selects lowest RMSE/highest $, and calculates empirical error ratio bands.
  * *Input/Output*: Train partition $\rightarrow$ candidate comparison table and best pipeline.
* **undler.py**:
  * *Operations*: Packages the pipeline, error bands, dependency versions, and vocabularies into .joblib.
  * *Input/Output*: Model artifacts $\rightarrow$ models/pipeline.joblib.

#### Module 4: Evaluation & Model Explainability (src/evaluation/)
* **metrics.py**:
  * *Operations*: Calculates RMSE, MAE, ^2$, or multi-class Precision/Recall/Macro-$ and confusion matrices.
  * *Input/Output*: y_true, y_pred $\rightarrow$ metrics dictionary and summary tables.
* **explainability.py**:
  * *Operations*: Computes global SHAP / Permutation feature importances for risk identification.
  * *Input/Output*: Test features $\rightarrow$ feature importance plots and intervention weights.

#### Module 5: Application & Serving Layer (pi/ & dashboard/)
* **pi/main.py**:
  * *Operations*: FastAPI endpoints (GET /health, POST /predict, POST /company-match).
  * *Input/Output*: Pydantic JSON requests $\rightarrow$ JSON predictions with error bands.
* **dashboard/app.py**:
  * *Operations*: Multi-page Streamlit portal (KPIs, EDA plots, interactive estimators, and resume/gap matching).
  * *Input/Output*: User UI input $\rightarrow$ visual charts, data tables, and advisory reports.


## 0. Global Setup & Dynamic Search Path Resolution

### Submodule Architecture & Path Ingestion
Before importing submodules, the project root and all submodule directories are injected into `sys.path`. We also initialize `%autoreload` so local Python package updates are hot-reloaded into kernel memory without restarting.

In [ ]:
import os
import sys
from pathlib import Path

# Resolve Project Root dynamically
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / 'src'

# Register subfolder module paths
for path_entry in [PROJECT_ROOT, SRC_DIR, SRC_DIR / 'preprocessing', SRC_DIR / 'features', SRC_DIR / 'models']:
    path_str = str(path_entry.resolve())
    if path_entry.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

print(f"[INIT] Project Root resolved: {PROJECT_ROOT}")
print(f"[INIT] Submodules path registered: {SRC_DIR}")

## 1. Submodule: `src.preprocessing` (Data Collection, Cleaning & Audit Trail)

### Scope & Responsibilities
The `src.preprocessing` submodule manages all initial data ingest, sanitation, and audit tracking. Key responsibilities include:
- **Multi-file Ingestion:** Combining raw CSV parts from `data/raw/` without row truncation.
- **Deduplication:** Removing exact duplicate job records (~900 rows in messy datasets).
- **Categorical Alias Normalization:** Standardizing city names (e.g., `BANGALORE` $\rightarrow$ `Bengaluru`, `Bombay` $\rightarrow$ `Mumbai`, `Gurgaon` $\rightarrow$ `Gurugram`).
- **Boundary Validation:** Constraining candidate experience years to $[0, 30]$ years and filtering out invalid or extreme salary outliers ($<0$ or $>60$ LPA).
- **Traceability:** Logging each transformation step in `data/processed/cleaning_log.csv` and `cleaning_log.md`.

In [ ]:
from src import preprocessing as prep

# 1. Load raw data and check initial dimensions
raw_df, source_names = prep.load_raw_data()
print(f"[RAW INGEST] Ingested files: {source_names}")
print(f"[RAW SHAPE] Dimensions: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")

# 2. Inspect raw data quality diagnostics
quality_metrics = prep.data_quality_report(raw_df)
print(f"[QUALITY] Exact duplicate rows: {quality_metrics['exact_duplicates']:,}")
print("[QUALITY] Missing values summary (Top 5):")
display(quality_metrics['missing'].head(5))

# 3. Execute deterministic cleaning pipeline and inspect audit log
cleaned_df, cleaning_audit_log = prep.clean_dataset(raw_df)
print(f"\n[CLEANED SHAPE] Retained rows: {len(cleaned_df):,} (from {len(raw_df):,})")
display(cleaning_audit_log)

# 4. Persist clean artifact and Markdown audit
prep.save_cleaned(cleaned_df, cleaning_audit_log, source_names)
print(f"[SAVED] Clean dataset persisted to: {prep.CLEANED_PATH}")

## 2. Submodule: `src.feature_engineering` (Custom Transformers & Pipeline Preprocessors)

### Scope & Responsibilities
The `src.feature_engineering` submodule implements scikit-learn compatible custom transformers:
- **`ExperienceFeatureExtractor`:** Parses textual experience strings (e.g., `'3-5 years'`, `'Fresher'`, `'10+ years'`) into numeric average experience, polynomial squared growth ($x^2$), and range width $(hi - lo)$.
- **`SkillBinarizer`:** Learns top $N$ in-demand skills strictly from the training partition to avoid target leakage, multi-hot encoding known competencies.
- **`build_preprocessor`:** Constructs a consolidated `ColumnTransformer` binding One-Hot Encoding for categorical features (`role`, `location`, `work_mode`, `education`) with numeric scaling.

In [ ]:
from src import feature_engineering as fe

# 1. Prepare standardized training frame (drop uncalibrated salaries and map column names)
prep.ensure_cleaned()
model_frame, prep_report = prep.prepare_for_model(prep.load_cleaned())
print(f"[FRAME REPORT] Model-ready rows: {prep_report['rows_out']:,}")
display(model_frame.head(3))

# 2. Build and verify the composite preprocessing pipeline
features_to_use = [col for col in ['role', 'experience', 'location', 'skills', 'work_mode', 'education'] if col in model_frame.columns]
preprocessor = fe.build_preprocessor(features=features_to_use, top_skills=40)

# Fit transformer on sample partition to inspect output feature schema
preprocessor.fit(model_frame[features_to_use].iloc[:500])
transformed_sample = preprocessor.transform(model_frame[features_to_use].iloc[:5])

print(f"[TRANSFORM VERIFICATION] Output matrix shape: {transformed_sample.shape}")
print(f"[FEATURE NAMES OUT] First 10 generated feature tokens:")
for feat_name in list(preprocessor.get_feature_names_out())[:10]:
    print(f"  • {feat_name}")

## 3. Submodule: `src.salary_model` (Model Selection, Cross-Validation & Range Estimation)

### Scope & Responsibilities
The `src.salary_model` submodule handles regression training, empirical interval calculation, and bundle persistence:
- **Strict Partitioning (`split_frame`):** Preserves an 80/20 train/test holdout before transformer fitting.
- **Candidate Regressors:** Evaluates Ridge, Decision Tree, Random Forest, Gradient Boosting, and XGBoost.
- **CV Model Selection:** Evaluates $K=5$ fold cross-validated RMSE and MAE on $\log(1+\text{salary})$ transformed targets.
- **Empirical Range Bands:** Calculates the 10th and 90th percentile prediction error ratios ($q_{10}, q_{90}$) to generate prediction intervals without assuming Gaussian errors.
- **Bundle Persistence:** Saves the model bundle (`salary_pipeline.joblib`) including Python/scikit-learn version checks and metadata.

In [ ]:
from src import salary_model as sm

# 1. Run model training and cross-validation across all candidate algorithms
print("[TRAINING] Training candidate regression models across 5-fold CV...")
bundle = sm.train_and_select(
    frame=model_frame,
    progress=lambda i, name: print(f"  [{i+1}] Training & evaluating {name}..."),
    data_label="cleaned_jobs.csv (practice dataset)"
)

# 2. Display model comparison metrics
print(f"\n[BEST MODEL SELECTED]: {bundle['best_name']}")
print(f"[EMPIRICAL ERROR BAND]: [{bundle['ratio_q10']:.2f}x, {bundle['ratio_q90']:.2f}x]")
display(bundle['results'])

# 3. Save model bundle artifact to disk
sm.save_bundle(bundle)
print(f"[PERSISTENCE] Bundle saved to: {sm.MODEL_PATH}")

## 4. Submodule: `src.skill_analyzer` (Role Benchmarks & Skill Gap Analysis)

### Scope & Responsibilities
The `src.skill_analyzer` submodule computes skill benchmarks and calculates match ratios:
- **Market Demand Mining:** Identifies standard in-demand skills for target job roles (e.g., Data Scientist, Data Engineer) directly from the cleaned dataset.
- **Gap Scoring:** Computes the skill match score: $\text{Score} = \frac{|\text{Resume} \cap \text{Role}|}{|\text{Role}|} \times 100$.
- **Actionable Guidance:** Recommends prioritized learning paths for missing competencies.

In [ ]:
from src import skill_analyzer as sa

# 1. Retrieve role benchmark for a target position
target_role = "Data Scientist"
role_ref = sa.get_role_skill_reference(target_role)
print(f"[BENCHMARK] Target Role: {target_role}")
print(f"[BENCHMARK] Market Postings Sampled: {role_ref['n_postings']}")
print(f"[BENCHMARK] In-Demand Skills: {role_ref['skills'][:8]}")

# 2. Compute skill gap against a candidate's resume skill list
candidate_skills = ["Python", "SQL", "Pandas", "NumPy"]
gap_analysis = sa.compute_skill_gap(
    skills=candidate_skills,
    ref_skills=role_ref["skills"],
    role=target_role
)

print(f"\n[GAP ANALYSIS SCORE]: {gap_analysis['score']:.1f}%")
print(f"[MATCHED SKILLS]: {gap_analysis['matched']}")
print(f"[MISSING SKILLS]: {gap_analysis['missing'][:5]}")

## 5. Submodule: `src.company_match` (Company Job Matching & SSRF Protection)

### Scope & Responsibilities
The `src.company_match` submodule safely compares a resume against external company job postings:
- **Safe URL Ingestion:** Enforces server-side request forgery (SSRF) protection by blocking loopback, private, link-local, and cloud metadata IP ranges, allowing only ports 80/443, honoring `robots.txt`, and bounding payloads to 2 MB.
- **Structured Data Extraction:** Parses schema.org `JobPosting` JSON-LD with fallback to clean HTML parsing.
- **Posting Fit Scoring:** Calculates matched skills, missing skills ranked by frequency of mentions, and extracted years of experience and educational requirements.

In [ ]:
from src import company_match as cm

# Mock job posting description text
sample_job_posting = """
Senior Data Scientist - Analytics Team
Key Requirements:
- 4+ years experience with Python, SQL, and Docker.
- Experience deploying Machine Learning pipelines on AWS or GCP.
- Strong background with PyTorch or TensorFlow for NLP applications.
- Master's degree in Computer Science or related quantitative field.
"""

# Analyze candidate fit
company_fit = cm.analyze_company_fit(
    resume_skills=candidate_skills,
    job_text=sample_job_posting
)

print(f"[COMPANY MATCH SCORE]: {company_fit['score']:.1f}%")
print(f"[POSTING SKILLS DETECTED]: {company_fit['job_skills']}")
print(f"[MATCHED SKILLS]: {company_fit['matched']}")
print(f"[MISSING SKILLS (RANKED BY MENTIONS)]:")
for s in company_fit['missing']:
    print(f"  • {s} (mentioned {company_fit['mentions'].get(s, 0)}x)")

print(f"\n[EXTRACTED REQUIREMENTS]")
print(f"  • Experience: {company_fit.get('years', 'Not specified')}")
print(f"  • Education: {company_fit.get('education', 'Not specified')}")

## 6. End-to-End Pipeline Inference Verification (`predict_salary`)

### Production Serving Verification
Finally, we execute inference through the full trained pipeline bundle to verify point predictions and empirical bounds.

In [ ]:
# Load persisted bundle and execute salary estimation
loaded_bundle = sm.load_bundle()

sample_query = {
    "role": "Data Scientist",
    "experience": 4.0,
    "location": "Bengaluru",
    "skills": "Python, SQL, Machine Learning, Docker",
    "work_mode": "Hybrid",
    "education": "Master's"
}

prediction_result = sm.predict_salary(loaded_bundle, **sample_query)

print("[PREDICTION RESULT]")
print(f"  • Point Estimate: INR {prediction_result['point']:,.0f} per year ({prediction_result['point']/100000:.2f} LPA)")
print(f"  • Expected Range: INR {prediction_result['low']:,.0f} - {prediction_result['high']:,.0f}")
if prediction_result.get('warnings'):
    print(f"  • Warnings: {prediction_result['warnings']}")

print("\n[DISCLAIMER]", prediction_result['disclaimer'])
print("\n✓ All submodules verified and executed successfully end-to-end.")